# 01 青空文庫 — データ取得・前処理・Word2Vec学習

**このノートブックの目的**

1. 青空文庫から孤独関連テキストを収集する
2. MeCab（fugashi）で形態素解析・トークナイズする
3. Word2Vecモデルを学習する
4. 孤独関連語の意味空間を可視化する

**分析の問い（Phase 1）**
- 「孤独」「孤立」「孤高」はどんな語と共起しているか？
- 時代（明治 / 大正 / 昭和）で意味空間は変わるか？
- Loneliness / Isolation / Solitude の方向ベクトルはデータから確認できるか？

---
> **注意**：青空文庫のテキストは著作権切れ作品が中心（概ね没後70年）。
> 明治〜昭和中期が主なカバー範囲になる。

## 0. 環境セットアップ

In [ ]:
# ライブラリのインストール（Colab用）
!pip install fugashi unidic-lite gensim umap-learn japanize-matplotlib jaconv -q
print('インストール完了')

In [ ]:
import os
import re
import time
import zipfile
import urllib.request
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語フォント対応
import seaborn as sns

import fugashi
import jaconv
from gensim.models import Word2Vec

# 再現性のためシードを固定
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# データ保存先（Colabの場合はGoogle Driveをマウントしてもよい）
DATA_DIR = Path('./data')
RAW_DIR = DATA_DIR / 'raw' / 'aozora'
PROCESSED_DIR = DATA_DIR / 'processed' / 'aozora'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('セットアップ完了')
print(f'  データ保存先: {RAW_DIR}')

---
## 1. 青空文庫からテキストを取得する

青空文庫はGitHubリポジトリ（`aozorabunko/aozorabunko`）に
全テキストが公開されています。

戦略：
- 作品CSVから作者の生没年を参照し、**時代ラベル**を付与
- 孤独研究に関連しそうな作家・ジャンルを意識しつつ、まずは広く取得
- 最初は**100〜200作品程度**で動作確認

In [ ]:
# 青空文庫の作品リスト（CSV）を取得する
# 公式配布: https://www.aozora.gr.jp/index_pages/person_all.html
# GitHubから作品リストCSVを取得

CATALOG_URL = 'https://www.aozora.gr.jp/index_pages/list_person_all_extended_utf8.zip'
CATALOG_ZIP = RAW_DIR / 'catalog.zip'
CATALOG_CSV = RAW_DIR / 'list_person_all_extended_utf8.csv'

if not CATALOG_CSV.exists():
    print('作品リストをダウンロード中...')
    urllib.request.urlretrieve(CATALOG_URL, CATALOG_ZIP)
    with zipfile.ZipFile(CATALOG_ZIP, 'r') as zf:
        zf.extractall(RAW_DIR)
    print('完了')
else:
    print('作品リストは取得済み')

# CSVを読み込む
df_catalog = pd.read_csv(CATALOG_CSV, encoding='utf-8')
print(f'総作品数: {len(df_catalog):,}')
df_catalog.head(3)

In [ ]:
# カラム名を確認
print('カラム一覧:')
for col in df_catalog.columns:
    print(f'  {col}')

In [ ]:
# 時代ラベルを付与する
# 「没年」が分かる作品に絞り、時代を分類する

def assign_era(birth_year_str):
    """生年から時代ラベルを返す（大雑把な区分）"""
    try:
        year = int(str(birth_year_str)[:4])  # 「1868年」などから4桁を取得
        if year < 1868:
            return '江戸以前'
        elif year < 1900:
            return '明治'
        elif year < 1912:
            return '明治後期〜大正'
        elif year < 1926:
            return '大正〜昭和初期'
        elif year < 1945:
            return '昭和戦前'
        else:
            return '昭和戦後以降'
    except:
        return '不明'

# 生年カラムを探す（カラム名がCatalogによって異なる場合があるため確認）
birth_col = [c for c in df_catalog.columns if '生年' in c or 'birth' in c.lower()]
print(f'生年関連カラム: {birth_col}')

if birth_col:
    df_catalog['era'] = df_catalog[birth_col[0]].apply(assign_era)
    print(df_catalog['era'].value_counts())

In [ ]:
# 公開されているテキストファイルのURLを絞り込む
# テキストファイル（.txt形式）が存在するものに絞る

# テキストURLのカラムを確認
url_cols = [c for c in df_catalog.columns if 'テキスト' in c or 'URL' in c or 'url' in c.lower()]
print(f'URL関連カラム: {url_cols}')

# 利用可能なテキスト数
txt_col = 'テキストファイルURL'  # カラム名が違う場合は上で確認して変更
if txt_col in df_catalog.columns:
    available = df_catalog[df_catalog[txt_col].notna()]
    print(f'テキストファイルあり: {len(available):,}件')

In [ ]:
# 分析対象を選定する
# まず「明治〜昭和」の日本語テキストを中心に100作品程度でテスト

# カタログの実際のカラム名に合わせて調整
# （上のセルで確認したカラム名を使う）

# --- ここを実際のカラム名に書き換える ---
TEXT_URL_COL = 'テキストファイルURL'   # テキストURLのカラム名
TITLE_COL = '作品名'                  # 作品名
AUTHOR_COL = '姓'                     # 著者姓
# ----------------------------------------

# テキストURLがあり、時代が「不明」でない作品に絞る
if 'era' in df_catalog.columns and TEXT_URL_COL in df_catalog.columns:
    df_target = df_catalog[
        df_catalog[TEXT_URL_COL].notna() &
        (df_catalog['era'] != '不明') &
        (df_catalog['era'] != '江戸以前')
    ].copy()
    print(f'対象作品数: {len(df_target):,}')
    print(df_target['era'].value_counts())

In [ ]:
# テキストをダウンロードする関数

def download_aozora_text(url, save_path):
    """
    青空文庫のテキストzipをダウンロードして展開し、
    テキスト内容を返す。
    """
    try:
        tmp_zip = str(save_path) + '.zip'
        urllib.request.urlretrieve(url, tmp_zip)
        
        with zipfile.ZipFile(tmp_zip, 'r') as zf:
            # .txtファイルを探す
            txt_files = [f for f in zf.namelist() if f.endswith('.txt')]
            if not txt_files:
                return None
            with zf.open(txt_files[0]) as f:
                # 青空文庫はShift-JISが多い
                try:
                    content = f.read().decode('shift-jis')
                except:
                    content = f.read().decode('utf-8', errors='ignore')
        
        os.remove(tmp_zip)
        return content
    except Exception as e:
        print(f'  エラー: {e}')
        return None


def clean_aozora_text(text):
    """
    青空文庫テキストの前処理：
    - ヘッダー・フッター除去
    - ルビ除去（《》内）
    - 注記除去（［］内）
    - 全角数字・記号の正規化
    """
    if text is None:
        return ''
    
    # ヘッダー（「-------」より前）を除去
    if '-------' in text:
        text = text.split('-------')[-1]
    
    # ルビ除去: 漢字《ルビ》 → 漢字
    text = re.sub(r'《[^》]*》', '', text)
    
    # 注記除去: ［＃注記内容］
    text = re.sub(r'［＃[^］]*］', '', text)
    
    # 縦棒（｜）除去（ルビの始まり記号）
    text = re.sub(r'｜', '', text)
    
    # 全角英数を半角に
    text = jaconv.z2h(text, kana=False, ascii=True, digit=True)
    
    # 改行を統一
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    
    # 連続する空行を1つに
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()


print('関数定義完了')

In [ ]:
# まず小規模サンプル（各時代20作品程度）でテスト
# 本番では全量取得

N_SAMPLE_PER_ERA = 20  # 時代ごとのサンプル数

if 'era' in df_catalog.columns and TEXT_URL_COL in df_catalog.columns:
    df_sample = (
        df_target
        .groupby('era', group_keys=False)
        .apply(lambda x: x.sample(min(N_SAMPLE_PER_ERA, len(x)), random_state=RANDOM_SEED))
    )
    print(f'サンプル数: {len(df_sample)}')
    print(df_sample['era'].value_counts())
    
    # ダウンロード実行
    corpus = []  # [(テキスト, 時代ラベル, 作品名), ...]
    
    for i, row in df_sample.iterrows():
        url = row[TEXT_URL_COL]
        era = row.get('era', '不明')
        title = row.get(TITLE_COL, f'作品{i}')
        author = row.get(AUTHOR_COL, '')
        
        print(f'  [{era}] {author}「{title}」 ...', end='')
        text = download_aozora_text(url, RAW_DIR / f'{i}.txt')
        cleaned = clean_aozora_text(text)
        
        if cleaned and len(cleaned) > 500:  # 短すぎるテキストは除外
            corpus.append({'text': cleaned, 'era': era, 'title': title, 'author': author})
            print(f' OK ({len(cleaned):,}文字)')
        else:
            print(' スキップ（短すぎ）')
        
        time.sleep(0.5)  # サーバーに負荷をかけない
    
    print(f'\n取得完了: {len(corpus)}作品')
    
    # DataFrameに変換して保存
    df_corpus = pd.DataFrame(corpus)
    df_corpus.to_parquet(PROCESSED_DIR / 'corpus_raw.parquet', index=False)
    print('保存完了:', PROCESSED_DIR / 'corpus_raw.parquet')

---
## 2. 形態素解析・トークナイズ

fugashi（MeCabのPythonラッパー）+ unidic-lite で解析。
Word2Vecの学習には**名詞・動詞・形容詞・副詞**を使う。

In [ ]:
# 形態素解析の設定
tagger = fugashi.Tagger()

# 対象品詞（Word2Vec学習に使う品詞）
TARGET_POS = {'名詞', '動詞', '形容詞', '副詞'}

def tokenize(text, pos_filter=TARGET_POS):
    """
    テキストをMeCabでトークナイズして、
    対象品詞の原形リストを返す。
    
    Word2Vecの学習には「文のリスト（各文が単語リスト）」が必要なため、
    ここでは句点で文を区切って返す。
    """
    sentences = []
    current = []
    
    for word in tagger(text):
        pos = word.feature.pos1  # 品詞
        surface = word.surface   # 表層形
        
        # 句点・感嘆符・疑問符で文区切り
        if surface in {'。', '！', '？', '\n\n'}:
            if len(current) >= 3:  # 短すぎる文は除外
                sentences.append(current)
            current = []
            continue
        
        # 対象品詞のみ
        if pos not in pos_filter:
            continue
        
        # 原形を取得（UniDicの場合）
        try:
            lemma = word.feature.lemma
            if lemma and lemma != '*':
                token = lemma
            else:
                token = surface
        except:
            token = surface
        
        # 1文字の語・数字・記号は除外
        if len(token) < 2 or token.isnumeric():
            continue
        
        current.append(token)
    
    if len(current) >= 3:
        sentences.append(current)
    
    return sentences


# 動作確認
test_text = '彼は孤独を感じていた。その孤立した日々の中で、ひとり静かに書き続けた。'
test_tokens = tokenize(test_text)
print('テスト結果:')
for sent in test_tokens:
    print(' ', sent)

In [ ]:
# コーパス全体をトークナイズ
# 時代ラベルごとに文リストを作成する

if 'df_corpus' not in dir():
    df_corpus = pd.read_parquet(PROCESSED_DIR / 'corpus_raw.parquet')

print('トークナイズ開始...')
all_sentences = []           # 全文（時代混合）
era_sentences = {}           # 時代別の文リスト

for _, row in df_corpus.iterrows():
    sentences = tokenize(row['text'])
    all_sentences.extend(sentences)
    
    era = row['era']
    if era not in era_sentences:
        era_sentences[era] = []
    era_sentences[era].extend(sentences)

print(f'総文数: {len(all_sentences):,}')
for era, sents in sorted(era_sentences.items()):
    print(f'  {era}: {len(sents):,}文')

---
## 3. Word2Vecモデルの学習

### なぜWord2Vecか

- 解釈しやすい（単語 → ベクトル → 類似語が取れる）
- 計算コストが低く、青空文庫規模でも十分学習できる
- 「孤独方向ベクトル」の試作に適している
- BCCWJ・YouTubeとの比較の出発点として扱いやすい

### パラメータの意味

| パラメータ | 値 | 意味 |
|------------|-----|------|
| `vector_size` | 100 | 埋め込み次元数（大きいほど表現力↑、データが必要） |
| `window` | 5 | 文脈窓（±5単語） |
| `min_count` | 3 | 最低出現回数（低頻度語を除外） |
| `sg` | 1 | 1=Skip-gram, 0=CBOW（小コーパスはSkip-gram推奨） |
| `epochs` | 10 | 学習エポック数 |

In [ ]:
# Word2Vecを全コーパスで学習

W2V_PARAMS = dict(
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,           # Skip-gram
    epochs=10,
    workers=4,
    seed=RANDOM_SEED,
)

print('Word2Vec学習中...')
model = Word2Vec(sentences=all_sentences, **W2V_PARAMS)

print(f'語彙数: {len(model.wv):,}')
model.save(str(PROCESSED_DIR / 'w2v_aozora_all.model'))
print('モデル保存完了')

In [ ]:
# 動作確認：孤独関連語の類似語を確認する

SEED_WORDS = ['孤独', '孤立', '孤高', '寂しい', 'ひとり', '一人']

for word in SEED_WORDS:
    if word in model.wv:
        similar = model.wv.most_similar(word, topn=10)
        print(f'\n【{word}】に近い語:')
        for w, score in similar:
            print(f'  {w:10s}  {score:.3f}')
    else:
        print(f'\n【{word}】: 語彙になし（出現回数不足の可能性）')

---
## 4. 孤独語の意味空間を可視化する

UMAPで高次元ベクトルを2次元に圧縮し、散布図を描く。

孤独関連語と「比較語」をあわせて表示することで、
意味空間上での孤独語のポジションが見えてくる。

In [ ]:
# 可視化対象の語彙を定義
# ---- 研究上の三分類に対応するSeed語彙 ----

LONELINESS_SEEDS = [
    '孤独', '寂しい', '悲しい', '虚しい', '空虚', '疎外', '取り残す',
    '孤独感', 'さみしい', '心細い'
]

ISOLATION_SEEDS = [
    '孤立', '断絶', '疎遠', '引きこもる', '排除', '遠ざかる',
    '孤立無援', 'ひとりぼっち', '無縁'
]

SOLITUDE_SEEDS = [
    '孤高', '独り', '静かだ', '自由', '内省', '沈黙', '瞑想',
    '一人', 'ひとり', '静謐'
]

# 比較用の「つながり語彙」
CONNECTION_SEEDS = [
    '友人', '家族', '絆', 'つながる', '共に', '仲間', '集う', '愛する'
]

# モデルの語彙に存在する語だけ使う
def filter_vocab(words, model):
    return [w for w in words if w in model.wv]

loneliness_words = filter_vocab(LONELINESS_SEEDS, model)
isolation_words  = filter_vocab(ISOLATION_SEEDS, model)
solitude_words   = filter_vocab(SOLITUDE_SEEDS, model)
connection_words = filter_vocab(CONNECTION_SEEDS, model)

print('可視化対象語数:')
print(f'  Loneliness: {len(loneliness_words)} / {len(LONELINESS_SEEDS)} → {loneliness_words}')
print(f'  Isolation:  {len(isolation_words)} / {len(ISOLATION_SEEDS)} → {isolation_words}')
print(f'  Solitude:   {len(solitude_words)} / {len(SOLITUDE_SEEDS)} → {solitude_words}')
print(f'  Connection: {len(connection_words)} / {len(CONNECTION_SEEDS)} → {connection_words}')

In [ ]:
import umap.umap_ as umap

# 可視化対象語をまとめる
viz_words = loneliness_words + isolation_words + solitude_words + connection_words
viz_labels = (
    ['Loneliness'] * len(loneliness_words) +
    ['Isolation'] * len(isolation_words) +
    ['Solitude'] * len(solitude_words) +
    ['Connection（比較）'] * len(connection_words)
)

if len(viz_words) < 5:
    print('語彙が少なすぎます。コーパスを増やすか、seed語彙を調整してください。')
else:
    # ベクトルを取得
    vectors = np.array([model.wv[w] for w in viz_words])
    
    # UMAP で2次元に圧縮
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=min(10, len(viz_words)-1),
        random_state=RANDOM_SEED,
        metric='cosine'
    )
    embedding = reducer.fit_transform(vectors)
    
    # 散布図を描く
    fig, ax = plt.subplots(figsize=(12, 9))
    
    COLORS = {
        'Loneliness': '#E74C3C',
        'Isolation':  '#3498DB',
        'Solitude':   '#2ECC71',
        'Connection（比較）': '#95A5A6'
    }
    MARKERS = {
        'Loneliness': 'o',
        'Isolation':  's',
        'Solitude':   '^',
        'Connection（比較）': 'x'
    }
    
    for label in set(viz_labels):
        idx = [i for i, l in enumerate(viz_labels) if l == label]
        ax.scatter(
            embedding[idx, 0], embedding[idx, 1],
            c=COLORS[label], marker=MARKERS[label],
            s=100, label=label, alpha=0.8
        )
    
    # 語ラベルを表示
    for i, (word, label) in enumerate(zip(viz_words, viz_labels)):
        ax.annotate(
            word,
            (embedding[i, 0], embedding[i, 1]),
            fontsize=10,
            textcoords='offset points',
            xytext=(4, 4)
        )
    
    ax.legend(fontsize=11)
    ax.set_title('孤独関連語の意味空間（青空文庫 Word2Vec + UMAP）', fontsize=14)
    ax.set_xlabel('UMAP次元1')
    ax.set_ylabel('UMAP次元2')
    plt.tight_layout()
    plt.savefig(PROCESSED_DIR / 'umap_loneliness_words.png', dpi=150)
    plt.show()
    print('図を保存しました')

---
## 5. 方向ベクトルの試作（Loneliness / Isolation / Solitude 軸）

### 考え方

指導教官の助言「3分類を教師ラベルではなく、意味空間上の方向ベクトルとして扱う」を実装する。

方向ベクトルの作り方：
1. 各カテゴリのSeed語彙のベクトルを**平均**する
2. それを「その方向への軸」として使う
3. 任意の語がその軸にどれくらい近いかを**コサイン類似度**で測る

例：「孤立」という語は Isolation軸に近いか、Loneliness軸に近いか？

In [ ]:
from numpy.linalg import norm

def make_axis_vector(seed_words, model):
    """Seed語のベクトル平均を軸ベクトルとして返す"""
    vecs = [model.wv[w] for w in seed_words if w in model.wv]
    if not vecs:
        return None
    axis = np.mean(vecs, axis=0)
    return axis / norm(axis)  # 正規化


def cosine_similarity(v1, v2):
    return float(np.dot(v1, v2) / (norm(v1) * norm(v2)))


def score_on_axis(word, axis_vector, model):
    """語がaxis方向にどれくらい向いているかを返す"""
    if word not in model.wv or axis_vector is None:
        return None
    wv = model.wv[word]
    return cosine_similarity(wv, axis_vector)


# 3軸を作成
axis_loneliness = make_axis_vector(loneliness_words, model)
axis_isolation  = make_axis_vector(isolation_words, model)
axis_solitude   = make_axis_vector(solitude_words, model)

print('軸ベクトル作成完了')
if axis_loneliness is not None:
    print(f'  Loneliness軸: {len(loneliness_words)}語から')
if axis_isolation is not None:
    print(f'  Isolation軸:  {len(isolation_words)}語から')
if axis_solitude is not None:
    print(f'  Solitude軸:   {len(solitude_words)}語から')

In [ ]:
# 孤独周辺語を3軸でスコアリング

TARGET_WORDS_FOR_SCORING = [
    '孤独', '孤立', '孤高', '寂しい', '独り', '一人',
    '悲しい', '自由', '静かだ', '断絶', '友人', '家族', '愛する',
    '夜', '部屋', '窓', '海', '空'
]

rows = []
for word in TARGET_WORDS_FOR_SCORING:
    if word not in model.wv:
        continue
    row = {
        'word': word,
        'Loneliness': score_on_axis(word, axis_loneliness, model),
        'Isolation':  score_on_axis(word, axis_isolation, model),
        'Solitude':   score_on_axis(word, axis_solitude, model),
    }
    rows.append(row)

df_scores = pd.DataFrame(rows).set_index('word')
print(df_scores.round(3).to_string())

In [ ]:
# ヒートマップで可視化

if not df_scores.empty:
    fig, ax = plt.subplots(figsize=(8, max(4, len(df_scores) * 0.5)))
    sns.heatmap(
        df_scores,
        annot=True, fmt='.2f',
        cmap='RdYlGn', center=0,
        vmin=-0.5, vmax=0.5,
        ax=ax
    )
    ax.set_title('孤独語の三軸スコア（Loneliness / Isolation / Solitude）', fontsize=13)
    plt.tight_layout()
    plt.savefig(PROCESSED_DIR / 'axis_scores_heatmap.png', dpi=150)
    plt.show()

---
## 6. 時代別比較（Word2Vecモデルの比較）

時代ごとに別々のWord2Vecモデルを学習し、
「孤独」の類似語がどう変化するかを確認する。

⚠️ 時代ごとのデータが少ない場合は学習が不安定になる。
BCCWJの本分析では安定するため、ここではあくまでパイプライン確認として扱う。

In [ ]:
# 時代別モデルを学習
era_models = {}

for era, sents in sorted(era_sentences.items()):
    if len(sents) < 100:  # データが少なすぎる場合はスキップ
        print(f'[{era}] データ不足でスキップ ({len(sents)}文)')
        continue
    
    print(f'[{era}] 学習中... ({len(sents):,}文)', end='')
    m = Word2Vec(sentences=sents, **W2V_PARAMS)
    era_models[era] = m
    print(f' 完了（語彙数: {len(m.wv):,}）')

print(f'\n学習済み時代: {list(era_models.keys())}')

In [ ]:
# 時代別の「孤独」類似語比較

FOCUS_WORD = '孤独'

for era, m in sorted(era_models.items()):
    if FOCUS_WORD in m.wv:
        similar = m.wv.most_similar(FOCUS_WORD, topn=10)
        print(f'\n【{era}】「{FOCUS_WORD}」に近い語:')
        for w, s in similar:
            print(f'  {w:12s} {s:.3f}')
    else:
        print(f'\n【{era}】「{FOCUS_WORD}」は語彙になし')

---
## メモ・気づき

（このセルに分析中の気づき・疑問点を随時記録する）

- 
- 
- 

## 次のステップ

- [ ] サンプル数を増やして再実行（全作品取得）
- [ ] Sentence-BERT に切り替えて文レベル分析
- [ ] 孤独語を含む文の抽出・クラスタリング
- [ ] BCCWJが届いたら同じパイプラインを適用